In [1]:
# Celda 1 - Conexión e imports para usuarios
from dao.mongo_dao import MongoDAO
from dao.usuario_dao import UsuarioDAO
from bson import ObjectId

mongo = MongoDAO()
db = mongo.connect()
usuario_dao = UsuarioDAO(mongo)
print("Conectado a DB:", mongo.db.name)


Conectado a DB: civictech


In [5]:
# Celda 2 - Crear usuario (ajustar municipio_id por uno válido)
nuevo_usuario = {
    "nombre_apellido": "María López",
    "dni": "87654321",
    "email": "maria.lopez@example.com",
    "contrasena": "MiPasswordSeguro123",   # se guardará hasheada en 'contrasena'
    "municipio_id": "6a21d09b4076ce51229df8a4",  # p.ej. "650f2a..."
    "telefono": "+54 9 11 0000 0000",
    "reputacion": 0
}

try:
    uid = usuario_dao.create_user(nuevo_usuario)
    print("✅ Usuario creado con _id:", uid)
except ValueError as e:
    print("⚠️ No se creó el usuario:", e)
except Exception as e:
    import traceback
    print("❌ Error inesperado:", type(e).__name__, "-", e)
    traceback.print_exc()


⚠️ No se creó el usuario: Ya existe un usuario con ese DNI


In [10]:
# Celda 3 - Listar usuarios mostrando el nombre del municipio
def print_table(rows, headers):
    cols = list(zip(*rows)) if rows else [[] for _ in headers]
    widths = []
    for i, h in enumerate(headers):
        col = cols[i] if i < len(cols) else []
        max_cell = max([len(str(x)) for x in col], default=0)
        widths.append(max(len(h), max_cell))
    header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep)
    for r in rows:
        print(" | ".join(str(r[i]).ljust(widths[i]) if i < len(r) else "".ljust(widths[i]) for i in range(len(headers))))

# Cargar usuarios
users = usuario_dao.list_users(limit=200)

if not users:
    print("No se encontraron usuarios.")
else:
    # Construir mapa de municipios: {_id: nombre}
    muni_ids = {u.get("municipio_id") for u in users if u.get("municipio_id") is not None}
    muni_map = {}
    if muni_ids:
        # Convertir a lista y buscar en la colección municipio
        # Acepta ObjectId o strings; la colección almacena ObjectId
        from bson import ObjectId
        query_ids = []
        for mid in muni_ids:
            try:
                query_ids.append(ObjectId(mid))
            except Exception:
                query_ids.append(mid)
        for m in mongo.db.municipio.find({"_id": {"$in": query_ids}}, {"nombre": 1}):
            muni_map[str(m["_id"])] = m.get("nombre")

    rows = []
    for u in users:
        mid = u.get("municipio_id")
        mid_str = str(mid) if mid is not None else ""
        muni_nombre = muni_map.get(mid_str, "(sin municipio)")
        rows.append([
            str(u.get("_id")),
            u.get("nombre_apellido",""),
            u.get("dni",""),
            u.get("email",""),
            muni_nombre,
            u.get("reputacion","")
        ])

    headers = ["_id", "nombre_apellido", "dni", "email", "municipio", "reputacion"]
    print_table(rows, headers)



_id                      | nombre_apellido | dni      | email                   | municipio        | reputacion
-------------------------+-----------------+----------+-------------------------+------------------+-----------
6a21d09b4076ce51229df8a6 | Juan Perez      | 30111222 | nuevo.email@example.com | Chilecito        | 50        
6a21d09b4076ce51229df8a7 | Maria Gomez     | 31111222 | maria.g@chilecito.com   | Chilecito        | 45        
6a21d09b4076ce51229df8a8 | Carlos Ruiz     | 32111222 | carlos.r@chilecito.com  | Chilecito        | 60        
6a21d09b4076ce51229df8a9 | Ana Torres      | 33111222 | ana.t@chilecito.com     | Chilecito        | 55        
6a21d09b4076ce51229df8ab | Marta Silva     | 35111222 | marta.s@larioja.com     | La Rioja Capital | 40        
6a21d09b4076ce51229df8ac | Pablo Lopez     | 36111222 | pablo.l@larioja.com     | La Rioja Capital | 70        
6a21d09b4076ce51229df8ad | Rosa Luna       | 37111222 | rosa.l@larioja.com      | La Rioja Capital | 65 

In [11]:
# Celda 4 - Actualizar email de un usuario y mostrar municipio (ejecutar después de listar)
# Si no reemplazás usuario_id, toma el primer usuario disponible.
usuario_id = "<REEMPLAZAR_POR_UN_ID_DE_USUARIO>"  # p.ej. "650f2a..."
nuevo_email = "nuevo.email@example.com"

# Si no reemplazaste, tomar el primer usuario
if usuario_id.startswith("<"):
    users = usuario_dao.list_users(limit=1)
    if not users:
        print("No hay usuarios disponibles para actualizar. Crea uno primero.")
        raise SystemExit
    usuario_id = str(users[0]["_id"])

def print_user_with_municipio(u):
    mid = u.get("municipio_id")
    muni_nombre = "(sin municipio)"
    if mid:
        try:
            from bson import ObjectId
            mid_obj = ObjectId(mid) if not isinstance(mid, ObjectId) else mid
        except Exception:
            mid_obj = mid
        muni = mongo.db.municipio.find_one({"_id": mid_obj}, {"nombre": 1})
        if muni:
            muni_nombre = muni.get("nombre", muni_nombre)
    print("Usuario _id:", u.get("_id"))
    print("  nombre_apellido:", u.get("nombre_apellido"))
    print("  dni:", u.get("dni"))
    print("  email actual:", u.get("email"))
    print("  municipio:", muni_nombre)

try:
    # Mostrar antes
    usuario_before = usuario_dao.find_by_dni(usuario_dao.collection.find_one({"_id": ObjectId(usuario_id)})["dni"]) \
        if False else usuario_dao.collection.find_one({"_id": ObjectId(usuario_id)})
    if not usuario_before:
        print("Usuario no encontrado con _id:", usuario_id)
    else:
        print("Antes de la actualización:")
        print_user_with_municipio(usuario_before)

        # Intentar actualizar
        try:
            mod = usuario_dao.update_email(usuario_id, nuevo_email)
            print("\nIntento de actualización. modified_count:", mod)
        except ValueError as e:
            print("\n⚠️ No se actualizó el email:", e)
        except Exception as e:
            print("\n❌ Error inesperado al actualizar email:", type(e).__name__, "-", e)
            raise

        # Mostrar después
        usuario_after = usuario_dao.collection.find_one({"_id": ObjectId(usuario_id)})
        print("\nDespués de la actualización:")
        print_user_with_municipio(usuario_after)

except ValueError as e:
    print("⚠️ Error de validación:", e)
except Exception as e:
    import traceback
    print("❌ Error inesperado:", type(e).__name__, "-", e)
    traceback.print_exc()


Antes de la actualización:
Usuario _id: 6a21d09b4076ce51229df8a6
  nombre_apellido: Juan Perez
  dni: 30111222
  email actual: nuevo.email@example.com
  municipio: Chilecito

Intento de actualización. modified_count: 0

Después de la actualización:
Usuario _id: 6a21d09b4076ce51229df8a6
  nombre_apellido: Juan Perez
  dni: 30111222
  email actual: nuevo.email@example.com
  municipio: Chilecito


In [9]:
# Celda 5 - Borrar usuario por _id y mostrar nombre del municipio
from bson import ObjectId

usuario_id = "6a21d09b4076ce51229df8aa"  # p.ej. "650f2a..."
# Si no reemplazaste, tomar el primer usuario
if usuario_id.startswith("<"):
    users = usuario_dao.list_users(limit=1)
    if not users:
        print("No hay usuarios disponibles para borrar. Crea uno primero.")
        raise SystemExit
    usuario_id = str(users[0]["_id"])

def print_user_with_municipio(u):
    mid = u.get("municipio_id")
    muni_nombre = "(sin municipio)"
    if mid:
        try:
            mid_obj = ObjectId(mid) if not isinstance(mid, ObjectId) else mid
        except Exception:
            mid_obj = mid
        muni = mongo.db.municipio.find_one({"_id": mid_obj}, {"nombre": 1})
        if muni:
            muni_nombre = muni.get("nombre", muni_nombre)
    print("Usuario _id:", u.get("_id"))
    print("  nombre_apellido:", u.get("nombre_apellido"))
    print("  dni:", u.get("dni"))
    print("  email:", u.get("email"))
    print("  municipio:", muni_nombre)

try:
    usuario = usuario_dao.collection.find_one({"_id": ObjectId(usuario_id)})
    if not usuario:
        print("Usuario no encontrado con _id:", usuario_id)
    else:
        print("Usuario encontrado:")
        print_user_with_municipio(usuario)

        confirm = True  # cambiar a False si querés pedir confirmación manual antes de borrar
        if confirm:
            deleted = usuario_dao.delete(usuario_id)
            if deleted:
                print("\n✅ Usuario borrado correctamente (deleted_count = 1).")
            else:
                print("\n⚠️ Intento de borrado realizado pero deleted_count = 0.")
        else:
            print("\nOperación de borrado cancelada por configuración (confirm=False).")
except ValueError as e:
    print("⚠️ Error de validación:", e)
except Exception as e:
    import traceback
    print("❌ Error inesperado:", type(e).__name__, "-", e)
    traceback.print_exc()


Usuario encontrado:
Usuario _id: 6a21d09b4076ce51229df8aa
  nombre_apellido: Luis Diaz
  dni: 34111222
  email: luis.d@larioja.com
  municipio: La Rioja Capital

✅ Usuario borrado correctamente (deleted_count = 1).
